In [23]:
import pandas as pd
import numpy as np
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [24]:
df_pool = pd.read_csv('composite_pool.csv')
df_nfa_tri = pd.read_csv('NFA_Monthly_Triangulated.csv')


In [25]:
master_df = pd.merge(df_nfa_tri, df_pool, on=['COUNTRY', 'Date'], how='inner')
print("Merged DataFrame Shape:", master_df.shape)
display(master_df.head())

Merged DataFrame Shape: (38316, 22)


,Date,NFA_Triangulated,COUNTRY,Exchange_Rate,CPI_VALUE,Export price index (EPI),Exports of goods,"Exports of goods, Price deflator","Exports of goods, Volume index",Import price index,...,"Imports of goods, Volume index",External_Debt,GDP,BOP_USD,FEDFUNDS,Monthly_Avg_VIXCLS,DXY_Index,Crude_Oil_Price,Gold_Price,COFER_Reserves
0,2006-01-01,93218.758449,Afghanistan,50.408385,28.892132,NaN,NaN,NaN,NaN,NaN,...,NaN,979344507.8,4.906148e+10,NaN,4.29,12.036000,100.000005,63.311818,549.860,5901.698911
1,2006-02-01,91517.798182,Afghanistan,49.813395,28.339968,NaN,NaN,NaN,NaN,NaN,...,NaN,979344507.8,4.906148e+10,NaN,4.49,12.471053,100.211170,60.805500,555.000,5901.698911
2,2006-03-01,90962.011546,Afghanistan,49.938065,27.688584,NaN,NaN,NaN,NaN,NaN,...,NaN,979344507.8,4.906148e+10,NaN,4.59,11.693913,100.428087,62.746957,557.090,5901.698911
3,2006-04-01,93750.179423,Afghanistan,49.705165,28.033718,NaN,NaN,NaN,NaN,NaN,...,NaN,979344507.8,4.906148e+10,NaN,4.79,11.847368,99.743480,70.340000,610.650,5901.698911
4,2006-05-01,95165.963962,Afghanistan,49.714195,27.349426,NaN,NaN,NaN,NaN,NaN,...,NaN,979344507.8,4.906148e+10,NaN,4.94,14.454545,97.511774,70.510000,675.393,5901.698911


In [26]:
# =================================================================
# PHASE 3: ADVANCED MACROECONOMIC FEATURE ENGINEERING (VECTORIZED)
# =================================================================
import pandas as pd
import numpy as np

print("⚙️ Engineering Advanced Ratios and Momentum (V4)...")

# Sort by Time first to ensure all mathematical rolling averages are accurate
master_df = master_df.sort_values(by=['COUNTRY', 'Date']).reset_index(drop=True)

# -------------------------------------------------------------
# 0. PRE-REQUISITES: Calculate Trade Balance 
# -------------------------------------------------------------
if 'Trade_Balance' not in master_df.columns and 'Exports of goods' in master_df.columns:
    master_df['Trade_Balance'] = master_df['Exports of goods'] - master_df['Imports of goods']

# -------------------------------------------------------------
# 1. NFA AND RESERVE COVERAGE (The Devaluation Buffer)
# -------------------------------------------------------------
master_df['NFA_to_GDP_Ratio'] = np.where(master_df['GDP'] > 0, master_df['NFA_Triangulated'] / master_df['GDP'], np.nan)
master_df['Import_Cover_Months'] = np.where(master_df['Imports of goods'] > 0, master_df['NFA_Triangulated'] / master_df['Imports of goods'], np.nan)
master_df['Debt_Coverage_Ratio'] = np.where(master_df['External_Debt'] > 0, master_df['NFA_Triangulated'] / master_df['External_Debt'], np.nan)

# -------------------------------------------------------------
# 2. ECONOMIC STRESS & DEBT RATIOS
# -------------------------------------------------------------
master_df['Debt_to_GDP_Ratio'] = np.where(master_df['GDP'] > 0, master_df['External_Debt'] / master_df['GDP'], np.nan)
master_df['Trade_Balance_to_GDP'] = np.where(master_df['GDP'] > 0, master_df['Trade_Balance'] / master_df['GDP'], np.nan)

# NFA-to-Debt Change Velocity (Grouped by country!)
delta_nfa = master_df.groupby('COUNTRY')['NFA_Triangulated'].diff(3)
delta_debt = master_df.groupby('COUNTRY')['External_Debt'].diff(3)
master_df['NFA_Debt_Velocity'] = np.where(delta_debt != 0, delta_nfa / delta_debt, np.nan)

# -------------------------------------------------------------
# 3. PRICE & YIELD SPREADS (Macro Divergence)
# -------------------------------------------------------------
master_df['Real_Capital_Drain'] = master_df['FEDFUNDS'] - master_df['CPI_VALUE']

# Fed Rate Momentum
rolling_fed = master_df.groupby('COUNTRY')['FEDFUNDS'].rolling(window=6, min_periods=1).mean().reset_index(level=0, drop=True)
master_df['Fed_Rate_Momentum'] = master_df['FEDFUNDS'] - rolling_fed

# -------------------------------------------------------------
# 4. COMMODITY EXPOSURE
# -------------------------------------------------------------
master_df['Oil_Sensitivity_Index'] = np.where(master_df['Crude_Oil_Price'] > 0, master_df['Trade_Balance'] / master_df['Crude_Oil_Price'], np.nan)
master_df['Gold_to_Debt_Ratio'] = np.where(master_df['External_Debt'] > 0, master_df['Gold_Price'] / master_df['External_Debt'], np.nan)

# -------------------------------------------------------------
# 5. ADVANCED STATISTICAL ANOMALY SIGNALS
# -------------------------------------------------------------
# NFA Z-Score 
rolling_mean_nfa = master_df.groupby('COUNTRY')['NFA_Triangulated'].rolling(window=12, min_periods=3).mean().reset_index(level=0, drop=True)
rolling_std_nfa = master_df.groupby('COUNTRY')['NFA_Triangulated'].rolling(window=12, min_periods=3).std().reset_index(level=0, drop=True)

master_df['NFA_Z_Score'] = np.where(rolling_std_nfa > 0, (master_df['NFA_Triangulated'] - rolling_mean_nfa) / rolling_std_nfa, np.nan)

# Exchange Rate Volatility 
master_df['Exchange_Rate_Volatility'] = master_df.groupby('COUNTRY')['Exchange_Rate'].rolling(window=3, min_periods=2).std().reset_index(level=0, drop=True)


print("✅ Advanced Feature Engineering Complete!")
print(f"Final ML Grid Shape: {master_df.shape}")
display(master_df[['COUNTRY', 'Date', 'Import_Cover_Months', 'NFA_Debt_Velocity', 'NFA_Z_Score']].tail())

⚙️ Engineering Advanced Ratios and Momentum (V4)...
✅ Advanced Feature Engineering Complete!
Final ML Grid Shape: (38316, 35)


,COUNTRY,Date,Import_Cover_Months,NFA_Debt_Velocity,NFA_Z_Score
38311,Zimbabwe,2025-08-01,NaN,NaN,1.012494
38312,Zimbabwe,2025-09-01,NaN,NaN,0.803741
38313,Zimbabwe,2025-10-01,NaN,NaN,0.527634
38314,Zimbabwe,2025-11-01,NaN,NaN,0.313211
38315,Zimbabwe,2025-12-01,NaN,NaN,-0.416370


In [27]:
# PHASE 4: THE TARGET VARIABLE & SYNTHETIC STRESS INDEX
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. THE SYNTHETIC COMPOSITE INDEX (GLOBAL MACRO STRESS)
# We select our core global variables. (Add 'DXY_Index' if it is in your dataset)
pca_features = ['Monthly_Avg_VIXCLS', 'FEDFUNDS', 'Crude_Oil_Price']
if 'DXY_Index' in master_df.columns:
    pca_features.append('DXY_Index')

# Isolate valid data to prevent NaNs from crashing the PCA math
valid_data = master_df[pca_features].dropna()

if len(valid_data) > 0:
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(valid_data)
    
    pca = PCA(n_components=1)
    pca_scores = pca.fit_transform(scaled_data).flatten()
    
    # Map the stress scores back to the main dataframe
    master_df.loc[valid_data.index, 'Synthetic_Stress_Index'] = pca_scores
else:
    master_df['Synthetic_Stress_Index'] = np.nan

# 2. EXTRA GLOBAL MOMENTUM (Gold & Dollar Squeeze)
if 'Gold_Price' in master_df.columns:
    master_df['Gold_3M_Change'] = master_df['Gold_Price'].pct_change(periods=3)

if 'DXY_Index' in master_df.columns:
    master_df['DXY_3M_Change'] = master_df['DXY_Index'].pct_change(periods=3)

# 3. THE TARGET VARIABLE (THE ANSWER KEY)
# Look 3 months into the FUTURE (-3) grouped by country
master_df['Future_Exchange_Rate_3M'] = master_df.groupby('COUNTRY')['Exchange_Rate'].shift(-3)

# Calculate the actual percentage change 
master_df['Exchange_Rate_Change'] = (master_df['Future_Exchange_Rate_3M'] - master_df['Exchange_Rate']) / master_df['Exchange_Rate']

# Define Crisis: A 5% (0.05) loss of currency value in 3 months
master_df['Crisis_Target'] = np.where(master_df['Exchange_Rate_Change'] >= 0.05, 1, 0)

# MASK THE FUTURE: The last 3 months of the dataset must be NaN, not 0!
master_df['Crisis_Target'] = np.where(master_df['Future_Exchange_Rate_3M'].isna(), np.nan, master_df['Crisis_Target'])

print("Answer Key & Composite Index Generated!")

# Let's see how many actual crises our dataset contains now!
total_crises = master_df['Crisis_Target'].sum()
print(f" Total Historical Crises Identified (Target=1): {int(total_crises)}")

# Show the final array of data ready for XGBoost
display(master_df[['COUNTRY', 'Date', 'Synthetic_Stress_Index', 'Exchange_Rate_Change', 'Crisis_Target']].tail(10))

Answer Key & Composite Index Generated!
 Total Historical Crises Identified (Target=1): 3750


,COUNTRY,Date,Synthetic_Stress_Index,Exchange_Rate_Change,Crisis_Target
38306,Zimbabwe,2025-03-01,1.837786,0.007749,0.0
38307,Zimbabwe,2025-04-01,1.699372,0.000092,0.0
38308,Zimbabwe,2025-05-01,1.823069,-0.004733,0.0
38309,Zimbabwe,2025-06-01,1.611674,-0.010221,0.0
38310,Zimbabwe,2025-07-01,1.618804,-0.011948,0.0
38311,Zimbabwe,2025-08-01,1.710215,-0.018757,0.0
38312,Zimbabwe,2025-09-01,1.645254,-0.023003,0.0
38313,Zimbabwe,2025-10-01,1.714999,NaN,NaN
38314,Zimbabwe,2025-11-01,1.689213,NaN,NaN
38315,Zimbabwe,2025-12-01,1.687180,NaN,NaN


In [28]:
master_df.head(20)
master_df.info()
master_df.shape
feature_names = master_df.columns.tolist()
print(feature_names)


<class 'pandas.DataFrame'>
RangeIndex: 38316 entries, 0 to 38315
Data columns (total 41 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Date                              38316 non-null  str    
 1   NFA_Triangulated                  38316 non-null  float64
 2   COUNTRY                           38316 non-null  str    
 3   Exchange_Rate                     38316 non-null  float64
 4   CPI_VALUE                         38316 non-null  float64
 5   Export price index (EPI)          4932 non-null   float64
 6   Exports of goods                  23208 non-null  float64
 7   Exports of goods, Price deflator  2820 non-null   float64
 8   Exports of goods, Volume index    300 non-null    float64
 9   Import price index                5520 non-null   float64
 10  Imports of goods                  23208 non-null  float64
 11  Imports of goods, Price deflator  2820 non-null   float64
 12  Imports of good

In [29]:
master_df.to_csv('final_master_df.csv', index=False)

In [30]:
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import xgboost as xgb
import numpy as np

In [31]:
# CLEAN THE TARGETS
# XGBoost can handle NaNs in the features, but it CANNOT handle NaNs in the Target variable which is the Answer Key!
ml_clean = master_df.dropna(subset=['Crisis_Target']).copy()
ml_clean.head()


,Date,NFA_Triangulated,COUNTRY,Exchange_Rate,CPI_VALUE,Export price index (EPI),Exports of goods,"Exports of goods, Price deflator","Exports of goods, Volume index",Import price index,...,Oil_Sensitivity_Index,Gold_to_Debt_Ratio,NFA_Z_Score,Exchange_Rate_Volatility,Synthetic_Stress_Index,Gold_3M_Change,DXY_3M_Change,Future_Exchange_Rate_3M,Exchange_Rate_Change,Crisis_Target
0,2006-01-01,93218.758449,Afghanistan,50.408385,28.892132,NaN,NaN,NaN,NaN,NaN,...,NaN,5.614572e-07,NaN,NaN,0.685723,NaN,NaN,49.705165,-0.013950,0.0
1,2006-02-01,91517.798182,Afghanistan,49.813395,28.339968,NaN,NaN,NaN,NaN,NaN,...,NaN,5.667056e-07,NaN,0.420721,0.797784,NaN,NaN,49.714195,-0.001991,0.0
2,2006-03-01,90962.011546,Afghanistan,49.938065,27.688584,NaN,NaN,NaN,NaN,NaN,...,NaN,5.688397e-07,-0.797337,0.313782,0.795536,NaN,NaN,49.982665,0.000893,0.0
3,2006-04-01,93750.179423,Afghanistan,49.705165,28.033718,NaN,NaN,NaN,NaN,NaN,...,NaN,6.235293e-07,1.040957,0.116547,0.614077,0.110555,-0.002565,50.082095,0.007583,0.0
4,2006-05-01,95165.963962,Afghanistan,49.714195,27.349426,NaN,NaN,NaN,NaN,NaN,...,NaN,6.896378e-07,1.315863,0.131935,0.482142,0.216924,-0.026937,50.004355,0.005837,0.0


In [32]:
# SELECT THE FEATURES (The Clues)
features = [
    # 1. Base Variables
    'Monthly_Avg_VIXCLS', 'FEDFUNDS', 'CPI_VALUE', 'Crude_Oil_Price',
    
    # 2. Devaluation Buffer Ratios
    'NFA_to_GDP_Ratio', 'Import_Cover_Months', 'Debt_Coverage_Ratio', 
    
    # 3. Debt & Trade Stress
    'Debt_to_GDP_Ratio', 'Trade_Balance_to_GDP', 'NFA_Debt_Velocity',
    
    # 4. Spreads & Divergence
    'Real_Capital_Drain', 'Fed_Rate_Momentum', 
    
    # 5. Commodity Exposure
    'Oil_Sensitivity_Index', 'Gold_to_Debt_Ratio',
    
    # 6. Statistical Anomalies
    'NFA_Z_Score', 'Exchange_Rate_Volatility',
    
    # 7. The Global Master Index
    'Synthetic_Stress_Index'
]
# Safely add Gold and DXY momentum if they successfully generated
if 'Gold_3M_Change' in ml_clean.columns: features.append('Gold_3M_Change')
if 'DXY_3M_Change' in ml_clean.columns: features.append('DXY_3M_Change')

print("✅ Feature List Locked In!")
print(f"Total features being sent to XGBoost: {len(features)}")

✅ Feature List Locked In!
Total features being sent to XGBoost: 19


In [33]:
# =================================================================
# STRICT TIME-BASED SPLIT (Preventing Target Leakage)
# =================================================================
# Ensure the Date column is a mathematical datetime object
ml_clean['Date'] = pd.to_datetime(ml_clean['Date'])

# Training Data: Learn from history (2000 - 2021)
# THE FIX: Added .dt.year to extract the mathematical year!
train_data = ml_clean[ml_clean['Date'].dt.year <= 2021]

# Testing Data: Simulate live trading (2022 - 2024)
test_data = ml_clean[ml_clean['Date'].dt.year > 2021]

X_train = train_data[features]
y_train = train_data['Crisis_Target']

X_test = test_data[features]
y_test = test_data['Crisis_Target']

print(f"📚 Training Rows (2000-2021): {len(X_train)}")
print(f"🔮 Testing Rows (2022-2024): {len(X_test)}")

📚 Training Rows (2000-2021): 32577
🔮 Testing Rows (2022-2024): 5319


In [34]:
# Calculate the ratio of Safe vs Crisis to balance the AI's attention
ratio = (len(y_train) - y_train.sum()) / y_train.sum()

In [35]:
model = xgb.XGBClassifier(
    n_estimators=300,        # How many decision trees to build
    max_depth=5,             # How deep the trees can think
    learning_rate=0.05,      # How fast it learns (slower prevents overfitting)
    scale_pos_weight=ratio,  # Forces the AI to pay extreme attention to Crises!
    random_state=42,
    eval_metric='auc',
    missing=np.nan           # Explicitly telling XGBoost "Don't panic if you see a NaN"
)
model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [36]:
#PREDICT AND EVALUATE ON THE 2022-2024
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1] # The exact percentage probability of a crash

In [37]:
print(classification_report(y_test, predictions, target_names=['Safe (0)', 'Crisis (1)']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, probabilities):.4f}")

              precision    recall  f1-score   support

    Safe (0)       0.90      0.99      0.94      4711
  Crisis (1)       0.53      0.12      0.19       608

    accuracy                           0.89      5319
   macro avg       0.71      0.55      0.56      5319
weighted avg       0.85      0.89      0.85      5319

ROC-AUC Score: 0.8050


In [38]:
#PHASE 5: HYPERPARAMETER TUNING (RANDOMIZED SEARCH)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
import numpy as np

In [39]:
# 1. Define the grid of brain configurations to test
param_grid = {
    'max_depth': [3, 4, 5, 6, 7, 8],               # How deep the logic goes
    'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.09, 0.1],   # How fast it updates its beliefs
    'n_estimators': [100, 200, 300, 500, 700, 900],       # Number of trees in the forest
    'subsample': [0.6, 0.8, 1.0, 1.1, 1.2],               # Prevents memorizing specific rows
    'colsample_bytree': [0.6, 0.8, 1.0, 1.1, 1.2]         # Forces the AI to use all features
}

In [40]:
# 2. Initialize the base model with our mandatory safety constraints
base_model = xgb.XGBClassifier(
    scale_pos_weight=ratio, 
    eval_metric='auc', 
    missing=np.nan,
    random_state=42
)

In [41]:
# 3. Setup the Automated Search (Testing 50 random configurations)
# We use cv=3 (Cross-Validation), meaning it tests each config 3 times to be sure!
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=50,
    scoring='roc_auc',
    cv=3, 
    verbose=1,
    random_state=42,
    n_jobs=-1 # Use all laptop CPU cores to run faster
)

In [42]:
print("Testing 50 different AI configurations.")
random_search.fit(X_train, y_train)

Testing 50 different AI configurations.
Fitting 3 folds for each of 50 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.8, ...], 'learning_rate': [0.01, 0.03, ...], 'max_depth': [3, 4, ...], 'n_estimators': [100, 200, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be us

In [43]:
# 4. Extract the ultimate winner
print("\n Tuning Complete!")
print("Best AI Parameters Found:")
for key, value in random_search.best_params_.items():
    print(f" - {key}: {value}")


 Tuning Complete!
Best AI Parameters Found:
 - subsample: 0.6
 - n_estimators: 300
 - max_depth: 7
 - learning_rate: 0.01
 - colsample_bytree: 0.6


In [44]:
# 5. Evaluate the optimized model on the unseen 2022-2024 Test Set
best_model = random_search.best_estimator_
opt_predictions = best_model.predict(X_test)
opt_probabilities = best_model.predict_proba(X_test)[:, 1]

print("\n OPTIMIZED MODEL PERFORMANCE (2022-2024)")
print("-" * 55)
print(classification_report(y_test, opt_predictions, target_names=['Safe (0)', 'Crisis (1)']))
print(f"Optimized ROC-AUC Score: {roc_auc_score(y_test, opt_probabilities):.4f}")


 OPTIMIZED MODEL PERFORMANCE (2022-2024)
-------------------------------------------------------
              precision    recall  f1-score   support

    Safe (0)       0.91      0.96      0.93      4711
  Crisis (1)       0.46      0.23      0.30       608

    accuracy                           0.88      5319
   macro avg       0.68      0.60      0.62      5319
weighted avg       0.86      0.88      0.86      5319

Optimized ROC-AUC Score: 0.8338
